In [1]:
# ============ 第二步:然后才导入其他库 ============
import random
import numpy as np
import torch
import pandas as pd
import pyterrier as pt
from pyterrier.measures import *
from collections import defaultdict
import ir_datasets
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

In [2]:
from typing import List
import pandas as pd

def get_easy_negatives(
    dataset, 
    qid: str, 
    df_A: pd.DataFrame,
    df_B: pd.DataFrame,
    num_negatives: int = 100,
    strategy: str = 'min'
) -> List[str]:
    """
    从两个系统中提取真正的"简单"负样本(两个系统都认为不相关)
    """
    qrels = dataset.get_qrels()
    
    # 筛选当前qid且label=0的文档
    neg_docs = qrels[(qrels['qid'] == qid) & (qrels['label'] == 0)].copy()
    
    if len(neg_docs) == 0:
        print(f"Warning: qid={qid} 没有找到任何负样本 (label=0)")
        return []
    
    # 获取当前qid的两个系统的结果
    df_A_qid = df_A[df_A['qid'] == qid].set_index('docno')
    df_B_qid = df_B[df_B['qid'] == qid].set_index('docno')
    
    # 为每个负样本计算综合score
    scores = []
    for docno in neg_docs['docno']:
        score_A = df_A_qid.loc[docno, 'score'] if docno in df_A_qid.index else 0
        score_B = df_B_qid.loc[docno, 'score'] if docno in df_B_qid.index else 0
        
        if strategy == 'min':
            combined_score = min(score_A, score_B)
        elif strategy == 'max':
            combined_score = max(score_A, score_B)
        elif strategy == 'mean':
            combined_score = (score_A + score_B) / 2
        else:
            combined_score = min(score_A, score_B)
        
        scores.append(combined_score)
    
    neg_docs['combined_score'] = scores
    
    # 按综合score升序排序(最低的在前)
    neg_docs_sorted = neg_docs.sort_values('combined_score', ascending=True)
    
    # 返回前num_negatives个docno
    return neg_docs_sorted['docno'].head(num_negatives).tolist()


def rank_based_interleaving_dedup_final(
    df_A: pd.DataFrame, 
    df_B: pd.DataFrame, 
    easy_negatives: dict or list,
    target_docs: int = 9,
    smart_easy_negative: bool = True
) -> pd.DataFrame:
    """
    基于文档级去重的交错算法 - 带标签回溯更新
    
    核心逻辑:
    1. 同步扫描 A 和 B 的位置 k
    2. 在位置 k 时,判断文档类型只基于**已扫描过的部分**
    3. 如果文档已被选过,检查是否需要**更新标签**为 Both
    4. 已选过的文档不重复添加(去重)
    """
    all_qids = set(df_A['qid'].unique()) | set(df_B['qid'].unique())
    final_results = []
    
    for qid in sorted(all_qids):
        # 1. 获取并排序
        list_A = df_A[df_A['qid'] == qid].copy()
        list_B = df_B[df_B['qid'] == qid].copy()
        
        list_A = list_A.sort_values('score', ascending=False).reset_index(drop=True)
        list_B = list_B.sort_values('score', ascending=False).reset_index(drop=True)
        
        # 2. 初始化
        query_selection = []
        seen_docs = {}  # 改为字典,存储 {docno: index_in_selection}
        count_both = 0
        count_a_only = 0
        count_b_only = 0
        
        # 3. 同步扫描两个排名
        k = 0
        max_k = max(len(list_A), len(list_B))
        
        while len(query_selection) < target_docs and k < max_k:
            # 获取第k位的文档
            doc_A = list_A.iloc[k]['docno'] if k < len(list_A) else None
            doc_B = list_B.iloc[k]['docno'] if k < len(list_B) else None
            
            # 创建当前已扫描部分的文档集合
            scanned_docs_A = set(list_A.iloc[:k+1]['docno']) if k < len(list_A) else set(list_A['docno'])
            scanned_docs_B = set(list_B.iloc[:k+1]['docno']) if k < len(list_B) else set(list_B['docno'])
            
            # 处理A的第k个文档
            if doc_A is not None:
                if doc_A not in seen_docs:
                    # 首次遇到这个文档
                    if doc_A in scanned_docs_B:
                        label = 'Both'
                        count_both += 1
                    else:
                        label = 'A-Only'
                        count_a_only += 1
                    
                    query_selection.append({
                        'qid': qid,
                        'docno': doc_A,
                        'origin_label': label
                    })
                    seen_docs[doc_A] = len(query_selection) - 1  # 记录位置
                    
                    # 检查是否已满
                    if len(query_selection) >= target_docs:
                        break
                else:
                    # 已经选过,检查是否需要更新标签
                    idx = seen_docs[doc_A]
                    current_label = query_selection[idx]['origin_label']
                    
                    # 如果当前是A-Only,但现在发现在B中也有 → 更新为Both
                    if current_label == 'A-Only' and doc_A in scanned_docs_B:
                        query_selection[idx]['origin_label'] = 'Both'
                        count_a_only -= 1
                        count_both += 1
                    # 如果当前是B-Only,但现在发现在A中也有 → 更新为Both
                    elif current_label == 'B-Only' and doc_A in scanned_docs_A:
                        query_selection[idx]['origin_label'] = 'Both'
                        count_b_only -= 1
                        count_both += 1
            
            # 处理B的第k个文档
            if doc_B is not None:
                if doc_B not in seen_docs:
                    # 首次遇到这个文档
                    if doc_B in scanned_docs_A:
                        label = 'Both'
                        count_both += 1
                    else:
                        label = 'B-Only'
                        count_b_only += 1
                    
                    query_selection.append({
                        'qid': qid,
                        'docno': doc_B,
                        'origin_label': label
                    })
                    seen_docs[doc_B] = len(query_selection) - 1  # 记录位置
                    
                    # 检查是否已满
                    if len(query_selection) >= target_docs:
                        break
                else:
                    # 已经选过,检查是否需要更新标签
                    idx = seen_docs[doc_B]
                    current_label = query_selection[idx]['origin_label']
                    
                    # 如果当前是B-Only,但现在发现在A中也有 → 更新为Both
                    if current_label == 'B-Only' and doc_B in scanned_docs_A:
                        query_selection[idx]['origin_label'] = 'Both'
                        count_b_only -= 1
                        count_both += 1
                    # 如果当前是A-Only,但现在发现在B中也有 → 更新为Both
                    elif current_label == 'A-Only' and doc_B in scanned_docs_B:
                        query_selection[idx]['origin_label'] = 'Both'
                        count_a_only -= 1
                        count_both += 1
            
            k += 1
        
        # 4. 智能Easy-Negative填充 (完整原始逻辑)
        if smart_easy_negative:
            remaining_slots = target_docs - count_both
            
            if remaining_slots % 2 == 1:
                needed_easy_negatives = 1
                
                if len(query_selection) >= target_docs:
                    # 移除最后一个A-Only或B-Only以保持平衡
                    if count_a_only > count_b_only:
                        for i in range(len(query_selection) - 1, -1, -1):
                            if query_selection[i]['origin_label'] == 'A-Only':
                                removed_doc = query_selection.pop(i)
                                del seen_docs[removed_doc['docno']]
                                count_a_only -= 1
                                break
                    else:
                        for i in range(len(query_selection) - 1, -1, -1):
                            if query_selection[i]['origin_label'] == 'B-Only':
                                removed_doc = query_selection.pop(i)
                                del seen_docs[removed_doc['docno']]
                                count_b_only -= 1
                                break
            else:
                needed_easy_negatives = 0
        else:
            needed_easy_negatives = target_docs - len(query_selection)
        
        # 5. 添加Easy-Negative
        if needed_easy_negatives > 0:
            if isinstance(easy_negatives, dict):
                neg_list = easy_negatives.get(qid, [])
            else:
                neg_list = easy_negatives
            
            added_negatives = 0
            for neg_doc in neg_list:
                if neg_doc not in seen_docs:
                    query_selection.append({
                        'qid': qid,
                        'docno': neg_doc,
                        'origin_label': 'Easy-Negative'
                    })
                    seen_docs[neg_doc] = len(query_selection) - 1
                    added_negatives += 1
                    
                    if added_negatives >= needed_easy_negatives:
                        break
            
            if added_negatives < needed_easy_negatives:
                print(f"Warning: qid={qid} 只填充了 {added_negatives} 个Easy-Negative,"
                      f"还缺少 {needed_easy_negatives - added_negatives} 个")
        
        # 6. 添加rank列
        for rank, doc_info in enumerate(query_selection, start=0):
            doc_info['rank'] = rank
        
        final_results.extend(query_selection)
    
    return pd.DataFrame(final_results)


def analyze_interleaving_results(df: pd.DataFrame) -> pd.DataFrame:
    """
    分析交错结果的统计信息
    """
    stats = []
    
    for qid in sorted(df['qid'].unique()):
        qid_df = df[df['qid'] == qid]
        
        label_counts = qid_df['origin_label'].value_counts()
        
        stats.append({
            'qid': qid,
            'total_docs': len(qid_df),
            'Both': label_counts.get('Both', 0),
            'A-Only': label_counts.get('A-Only', 0),
            'B-Only': label_counts.get('B-Only', 0),
            'Easy-Negative': label_counts.get('Easy-Negative', 0)
        })
    
    stats_df = pd.DataFrame(stats)
    
    # 添加汇总行
    total_row = pd.DataFrame([{
        'qid': 'TOTAL',
        'total_docs': stats_df['total_docs'].sum(),
        'Both': stats_df['Both'].sum(),
        'A-Only': stats_df['A-Only'].sum(),
        'B-Only': stats_df['B-Only'].sum(),
        'Easy-Negative': stats_df['Easy-Negative'].sum()
    }])
    
    return pd.concat([stats_df, total_row], ignore_index=True)


def add_query_text(df: pd.DataFrame, dataset_name: str = "msmarco-passage/trec-dl-2019/judged") -> pd.DataFrame:
    """
    为DataFrame添加query_text列
    """
    irds_dataset = ir_datasets.load(dataset_name)
    
    # 创建qid到query_text的映射
    query_map = {}
    for query in irds_dataset.queries_iter():
        query_map[str(query.query_id)] = query.text
    
    # 添加query_text列
    df['query_text'] = df['qid'].map(query_map)
    
    return df

In [3]:
# ============ 新增功能: 基于Preference的选择策略 ============

def select_by_preference(
    df_A: pd.DataFrame,
    df_B: pd.DataFrame,
    preference_df: pd.DataFrame,
    easy_negatives: dict,
    target_docs: int = 9,
    smart_easy_negative: bool = True
) -> pd.DataFrame:
    """
    基于preference.csv的选择策略:
    - PRF-Hurt: 仅使用ColBERT E2E的Top-9
    - PRF-Benefit: 仅使用ColBERT-PRF的Top-9
    - Neutral: 使用Team Draft Interleaving
    
    参数:
        df_A: ColBERT E2E (Baseline) 的排序结果
        df_B: ColBERT-PRF 的排序结果
        preference_df: 包含qid和preference列的DataFrame
        easy_negatives: 负样本字典
        target_docs: 目标文档数量
        smart_easy_negative: 是否智能补充负样本
    
    返回:
        pd.DataFrame: 合并后的结果
    """
    # 确保qid类型一致
    preference_df['qid'] = preference_df['qid'].astype(str)
    
    # 创建preference映射
    preference_map = preference_df.set_index('qid')['preference'].to_dict()
    
    all_qids = set(df_A['qid'].unique()) | set(df_B['qid'].unique())
    
    # 按preference分类qids (Insufficient-Data视为Neutral)
    prf_hurt_qids = [qid for qid in all_qids if preference_map.get(qid) == 'PRF-Hurt']
    prf_benefit_qids = [qid for qid in all_qids if preference_map.get(qid) == 'PRF-Benefit']
    neutral_qids = [qid for qid in all_qids if preference_map.get(qid) in ['Neutral', 'Insufficient-Data']]
    
    print(f"\nPreference Distribution:")
    print(f"  PRF-Hurt: {len(prf_hurt_qids)} queries")
    print(f"  PRF-Benefit: {len(prf_benefit_qids)} queries")
    print(f"  Neutral (including Insufficient-Data): {len(neutral_qids)} queries")
    
    results = []
    
    # 1. PRF-Hurt: 使用ColBERT E2E的Top-9
    if len(prf_hurt_qids) > 0:
        print("\nProcessing PRF-Hurt queries (using ColBERT E2E Top-9)...")
        for qid in tqdm(prf_hurt_qids, desc="PRF-Hurt"):
            df_qid = df_A[df_A['qid'] == qid].copy()
            df_qid = df_qid.sort_values('score', ascending=False).head(target_docs).reset_index(drop=True)
            df_qid['rank'] = range(len(df_qid))
            df_qid['origin_label'] = 'E2E-Only'  # 标记为仅来自E2E
            results.append(df_qid[['qid', 'docno', 'rank', 'origin_label']])
    
    # 2. PRF-Benefit: 使用ColBERT-PRF的Top-9
    if len(prf_benefit_qids) > 0:
        print("\nProcessing PRF-Benefit queries (using ColBERT-PRF Top-9)...")
        for qid in tqdm(prf_benefit_qids, desc="PRF-Benefit"):
            df_qid = df_B[df_B['qid'] == qid].copy()
            df_qid = df_qid.sort_values('score', ascending=False).head(target_docs).reset_index(drop=True)
            df_qid['rank'] = range(len(df_qid))
            df_qid['origin_label'] = 'PRF-Only'  # 标记为仅来自PRF
            results.append(df_qid[['qid', 'docno', 'rank', 'origin_label']])
    
    # 3. Neutral: 使用Team Draft Interleaving
    if len(neutral_qids) > 0:
        print("\nProcessing Neutral queries (using Team Draft Interleaving)...")
        df_A_neutral = df_A[df_A['qid'].isin(neutral_qids)]
        df_B_neutral = df_B[df_B['qid'].isin(neutral_qids)]
        
        neutral_result = rank_based_interleaving_dedup_final(
            df_A=df_A_neutral,
            df_B=df_B_neutral,
            easy_negatives=easy_negatives,
            target_docs=target_docs,
            smart_easy_negative=smart_easy_negative
        )
        results.append(neutral_result[['qid', 'docno', 'rank', 'origin_label']])
    
    # 合并所有结果
    final_result = pd.concat(results, ignore_index=True)
    
    # 按qid排序（数字顺序），然后按rank排序
    try:
        final_result['qid_numeric'] = final_result['qid'].astype(int)
        final_result = final_result.sort_values(['qid_numeric', 'rank']).drop('qid_numeric', axis=1)
    except:
        final_result = final_result.sort_values(['qid', 'rank'])
    
    final_result = final_result.reset_index(drop=True)
    
    return final_result

In [5]:
print("="*60)
print("COMPLETE PIPELINE WITH PREFERENCE")
print("="*60)

# 1. 读取去重后的数据
print("\n1. Loading deduplicated data...")
df_colbert = pd.read_csv('df_colbert_deduped.csv')
df_colbert_prf = pd.read_csv('df_colbert_prf_deduped.csv')

df_colbert['qid'] = df_colbert['qid'].astype(str)
df_colbert['docno'] = df_colbert['docno'].astype(str)
df_colbert_prf['qid'] = df_colbert_prf['qid'].astype(str)
df_colbert_prf['docno'] = df_colbert_prf['docno'].astype(str)

print(f"  ColBERT E2E: {len(df_colbert)} rows")
print(f"  ColBERT-PRF: {len(df_colbert_prf)} rows")

# 2. 读取preference.csv
print("\n2. Loading preference data...")
preference_df = pd.read_csv('preference.csv')
preference_df['qid'] = preference_df['qid'].astype(str)
print(f"  Loaded {len(preference_df)} query preferences")
print(f"  Preference distribution:")
print(preference_df['preference'].value_counts())

# 3. 准备dataset (改进的初始化方法)
print("\n3. Initializing PyTerrier and loading dataset...")
try:
    
    print("  Loading dataset...")
    dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2019/judged')
    print("  ✓ Dataset loaded successfully")
    
except Exception as e:
    print(f"  ⚠️ PyTerrier initialization failed: {e}")
    print("  Using ir_datasets as fallback...")
    
    # Fallback: 使用ir_datasets直接构建
    import ir_datasets as irds
    irds_dataset = irds.load('msmarco-passage/trec-dl-2019/judged')
    
    # 手动构建qrels DataFrame
    qrels_list = []
    for qrel in irds_dataset.qrels_iter():
        qrels_list.append({
            'qid': str(qrel.query_id),
            'docno': str(qrel.doc_id),
            'label': qrel.relevance
        })
    
    # 创建简单的dataset对象
    class SimpleDataset:
        def __init__(self, qrels_df):
            self._qrels = qrels_df
        
        def get_qrels(self):
            return self._qrels
    
    dataset = SimpleDataset(pd.DataFrame(qrels_list))
    print("  ✓ Fallback dataset created successfully")

# 4. 提取 easy negatives (仅用于Neutral查询)
print("\n4. Extracting easy negatives...")
all_qids = set(df_colbert['qid'].unique()) | set(df_colbert_prf['qid'].unique())
easy_negatives = {}

for qid in tqdm(all_qids, desc="Processing queries"):
    easy_negatives[qid] = get_easy_negatives(
        dataset=dataset,
        qid=qid,
        df_A=df_colbert,
        df_B=df_colbert_prf,
        num_negatives=100,
        strategy='min'
    )

# 5. 运行基于Preference的选择
print("\n5. Running preference-based selection...")
result = select_by_preference(
    df_A=df_colbert,
    df_B=df_colbert_prf,
    preference_df=preference_df,
    easy_negatives=easy_negatives,
    target_docs=9,
    smart_easy_negative=True
)

print(f"\n  Generated {len(result)} rows")

# 6. 分析结果
print("\n6. Analyzing results...")
stats = analyze_interleaving_results(result)
print("\n" + stats.to_string())

# 7. 添加passage_text和query_text
print("\n7. Adding text columns...")

# 7.1 从去重后的df中获取passage_text
passage_map_prf = df_colbert_prf.set_index('docno')['passage_text'].to_dict()
passage_map_e2e = df_colbert.set_index('docno')['passage_text'].to_dict()
passage_map = {**passage_map_prf, **passage_map_e2e}

result['passage_text'] = result['docno'].map(passage_map)

# 7.2 检查缺失的passage_text(主要是Easy-Negative)
missing_docnos = result[result['passage_text'].isna()]['docno'].unique()

if len(missing_docnos) > 0:
    print(f"\n⚠️  Found {len(missing_docnos)} documents without passage_text")
    print(f"   These are likely Easy-Negatives not in the original dfs")
    print(f"   Loading these documents from ir_datasets...")
    
    # 加载缺失的文档
    irds_dataset = ir_datasets.load("msmarco-passage/trec-dl-2019/judged")
    missing_doc_map = {}
    
    for doc in tqdm(irds_dataset.docs_iter(), total=8841823, desc="Loading missing docs"):
        doc_id_str = str(doc.doc_id)
        if doc_id_str in missing_docnos:
            missing_doc_map[doc_id_str] = doc.text
            if len(missing_doc_map) == len(missing_docnos):
                print(f"\n✓ Found all {len(missing_docnos)} missing documents!")
                break
    
    # 更新缺失的passage_text
    for idx, row in result[result['passage_text'].isna()].iterrows():
        if row['docno'] in missing_doc_map:
            result.at[idx, 'passage_text'] = missing_doc_map[row['docno']]
    
    print(f"Updated {len(missing_doc_map)} missing passages")

# 7.3 添加query_text
result = add_query_text(result, dataset_name="msmarco-passage/trec-dl-2019/judged")

# 调整列顺序
result = result[['qid', 'docno', 'rank', 'origin_label', 'query_text', 'passage_text']]

# 重命名列以符合服务器要求
result = result.rename(columns={
    'query_text': 'query',
    'passage_text': 'text'
})

# 按qid排序（数字顺序），然后按rank排序
print("\n  Sorting by qid and rank...")
try:
    result['qid_numeric'] = result['qid'].astype(int)
    result = result.sort_values(['qid_numeric', 'rank']).drop('qid_numeric', axis=1)
except:
    result = result.sort_values(['qid', 'rank'])

result = result.reset_index(drop=True)

# 最终列顺序
result = result[['qid', 'docno', 'rank', 'origin_label', 'query', 'text']]

# 8. 保存结果
print("\n8. Saving results...")
result.to_csv('result_preference_based_with_text.csv', index=False)
result.to_pickle('result_preference_based_with_text.pkl')

print("\nSaved to:")
print("  - result_preference_based_with_text.csv")
print("  - result_preference_based_with_text.pkl")

# 9. 显示样例
print("\n" + "="*60)
print("SAMPLE RESULTS:")
print("="*60)

for idx, row in result.head(3).iterrows():
    print(f"\n{idx+1}. Query {row['qid']} (Rank {row['rank']}, {row['origin_label']})")
    print(f"   Query: {row['query']}")
    print(f"   DocNo: {row['docno']}")
    print(f"   Passage: {row['text'][:120] if pd.notna(row['text']) else 'N/A'}...")

# 10. 数据质量检查
print("\n" + "="*60)
print("DATA QUALITY CHECK:")
print("="*60)
print(f"Total rows: {len(result)}")
print(f"Missing query: {result['query'].isna().sum()}")
print(f"Missing text: {result['text'].isna().sum()}")
print(f"Complete rows: {len(result.dropna())}")

# 查看origin_label分布
print("\nOrigin Label Distribution:")
print(result['origin_label'].value_counts())

print("\n" + "="*60)
print("✅ PIPELINE COMPLETE!")
print("="*60)

COMPLETE PIPELINE WITH PREFERENCE

1. Loading deduplicated data...
  ColBERT E2E: 281534 rows
  ColBERT-PRF: 37996 rows

2. Loading preference data...
  Loaded 43 query preferences
  Preference distribution:
preference
Neutral              21
PRF-Hurt             11
PRF-Benefit           9
Insufficient-Data     2
Name: count, dtype: int64

3. Initializing PyTerrier and loading dataset...
  Loading dataset...
  ✓ Dataset loaded successfully

4. Extracting easy negatives...


Processing queries: 100%|██████████| 43/43 [00:02<00:00, 15.87it/s]



5. Running preference-based selection...

Preference Distribution:
  PRF-Hurt: 11 queries
  PRF-Benefit: 9 queries
  Neutral (including Insufficient-Data): 23 queries

Processing PRF-Hurt queries (using ColBERT E2E Top-9)...


PRF-Hurt: 100%|██████████| 11/11 [00:00<00:00, 45.03it/s]



Processing PRF-Benefit queries (using ColBERT-PRF Top-9)...


PRF-Benefit: 100%|██████████| 9/9 [00:00<00:00, 215.55it/s]



Processing Neutral queries (using Team Draft Interleaving)...

  Generated 387 rows

6. Analyzing results...

        qid  total_docs  Both  A-Only  B-Only  Easy-Negative
0   1037798           9     0       0       0              0
1    104861           9     0       0       0              0
2   1063750           9     0       0       0              0
3   1103812           9     0       0       0              0
4   1106007           9     4       2       2              1
5   1110199           9     0       0       0              0
6   1112341           9     0       0       0              0
7   1113437           9     2       3       3              1
8   1114646           9     5       2       2              0
9   1114819           9     4       2       2              1
10  1115776           9     0       0       0              0
11  1117099           9     5       2       2              0
12  1121402           9     8       0       0              1
13  1121709           9     7      

Loading missing docs:  55%|█████▍    | 4841345/8841823 [01:09<00:57, 69527.42it/s]



✓ Found all 9 missing documents!
Updated 9 missing passages

  Sorting by qid and rank...

8. Saving results...

Saved to:
  - result_preference_based_with_text.csv
  - result_preference_based_with_text.pkl

SAMPLE RESULTS:

1. Query 19335 (Rank 0, PRF-Only)
   Query: anthropological definition of environment
   DocNo: 2304005
   Passage: Definition of environment - the surroundings or conditions in which a person, animal, or plant lives or operates, the na...

2. Query 19335 (Rank 1, PRF-Only)
   Query: anthropological definition of environment
   DocNo: 6512137
   Passage: 1 Meaning of environment. Environment means everything around to a living being. Especially the circumstances of life of...

3. Query 19335 (Rank 2, PRF-Only)
   Query: anthropological definition of environment
   DocNo: 6512144
   Passage: 3 Concept of environment. The environment is a system consisting of natural and artificial elements that are interrelate...

DATA QUALITY CHECK:
Total rows: 387
Missing query: 